# 🏗️ 기본 세팅

## MongoDB 서버 및 Python 패키지 설치

In [ ]:
# # 1. MongoDB 공식 GPG 키 및 리포지토리 등록
# !curl -fsSL https://www.mongodb.org/static/pgp/server-7.0.asc | sudo gpg --dearmor -o /usr/share/keyrings/mongodb-server-7.0.gpg
# !echo "deb [ arch=amd64,arm64 signed-by=/usr/share/keyrings/mongodb-server-7.0.gpg ] https://repo.mongodb.org/apt/ubuntu jammy/mongodb-org/7.0 multiverse" | sudo tee /etc/apt/sources.list.d/mongodb-org-7.0.list
#
# # 2. 패키지 목록 업데이트 및 설치
# !apt-get update
# !apt-get install -y mongodb-org > /dev/null
#
# # 3. MongoDB 서비스 실행
# !mkdir -p /data/db
# !mongod --fork --logpath /var/log/mongodb.log --dbpath /data/db
#
# # 4. 파이썬용 드라이버 설치
# !pip install pymongo

## MongoDB 접속 정보 넣기

In [1]:
import os
from pymongo import MongoClient
from bson import json_util  # $oid, $numberInt 등을 처리하기 위해 필수!

# 1. 로컬 MongoDB 연결
client = MongoClient('mongodb://localhost:27017/')
db = client.sample_mflix
movies = db.movies

# 2. 데이터 다운로드 (Raw 주소)
url = "https://raw.githubusercontent.com/neelabalan/mongodb-sample-dataset/main/sample_mflix/movies.json"
print("데이터 다운로드 중...")
os.system(f"wget -q -O movies.json {url}")

# 3. 데이터 파싱 및 DB 삽입
if os.path.exists('movies.json'):
    with open('movies.json', 'r') as f:
        try:
            # json_util을 사용하여 $oid 등의 특수 형식을 변환하며 읽기
            data = [json_util.loads(line) for line in f if line.strip()]

            # 중복 삽입 방지 및 데이터 주입
            if movies.count_documents({}) == 0:
                movies.insert_many(data)
                print(f"✅ 로드 성공! 총 {movies.count_documents({})}개의 문서가 저장되었습니다.")
            else:
                print(f"ℹ️ 이미 {movies.count_documents({})}개의 데이터가 DB에 있습니다.")
        except Exception as e:
            print(f"❌ 데이터 처리 중 오류 발생: {e}")
else:
    print("❌ movies.json 파일을 찾을 수 없습니다.")

# 4. 확인용 쿼리
sample = movies.find_one()
if sample:
    print(f"\n🔍 확인용 데이터:")
    print(f"제목: {sample.get('title')}")
    print(f"ID타입: {type(sample.get('_id'))}") # <class 'bson.objectid.ObjectId'> 로 나옴

데이터 다운로드 중...
✅ 로드 성공! 총 23539개의 문서가 저장되었습니다.

🔍 확인용 데이터:
제목: Blacksmith Scene
ID타입: <class 'bson.objectid.ObjectId'>


# 📝 연습하기

## 컬렉션 사용 준비

`sample_mflix` 데이터베이스의 `movies` 컬렉션은 이미 준비되어 있습니다.

삽입, 업데이트, 삭제와 같은 연습을 위해 `temp_movies`라는 임시 컬렉션을 생성하여 사용할 예정이며, 컬렉션은 첫 도큐먼트가 삽입될 때 자동으로 생성됩니다.

In [2]:
# 여기에 코드를 입력하세요.
temp_collection = db.temp_movies
print("완료")

완료


## 도큐먼트 삽입

### `insert_one()`: 단일 도큐먼트 삽입

`insert_one()` 메서드를 사용하여 컬렉션에 하나의 도큐먼트를 삽입합니다. 삽입된 도큐먼트의 `_id`를 반환받을 수 있습니다.

In [4]:
# 여기에 코드를 입력하세요.

test_mv= {
    "title":"OZ AH05",
    "year":"2026",
    "genre":["Comedy","Action"]
}
temp_collection.insert_one(test_mv)

InsertOneResult(ObjectId('69e9b033a83931b7031f0266'), acknowledged=True)

### `insert_many()`: 여러 도큐먼트 삽입

`insert_many()` 메서드를 사용하여 여러 도큐먼트를 리스트 형태로 한 번에 삽입합니다. 삽입된 모든 도큐먼트의 `_id` 리스트를 반환받을 수 있습니다.

In [5]:
# 여기에 코드를 입력하세요.
test_mv= [
    {
        "title":"OZ AH01",
        "year":"2025",
        "genre":["Comedy","Romance"]
    },
    {
        "title":"OZ AH02",
        "year":"2024",
        "genre":["Comedy","Horror"]
    },
    {
        "title":"OZ AH03",
        "year":"2023",
        "genre":["Fantasy","Action"]
    },
    {
        "title":"OZ AH04",
        "year":"2022",
        "genre":["comedy","Action"]
    },
]
temp_collection.insert_many(test_mv)

InsertManyResult([ObjectId('69e9b24ea83931b7031f0267'), ObjectId('69e9b24ea83931b7031f0268'), ObjectId('69e9b24ea83931b7031f0269'), ObjectId('69e9b24ea83931b7031f026a')], acknowledged=True)

## 데이터 조회

### `find_one()`: 조건에 맞는 첫 번째 도큐먼트 조회

`find_one()` 메서드는 주어진 조건과 일치하는 첫 번째 도큐먼트를 반환합니다. 조건이 없으면 컬렉션의 첫 번째 도큐먼트를 반환합니다.

In [11]:
# 여기에 코드를 입력하세요.
import pandas as pd
result = temp_collection.find_one({"title":"OZ AH05"})
display(pd.DataFrame([result]))

,_id,title,year,genre
0,69e9b033a83931b7031f0266,OZ AH05,2026,"[Comedy, Action]"


### `find()`: 조건에 맞는 모든 도큐먼트 조회

`find()` 메서드는 주어진 조건과 일치하는 모든 도큐먼트를 반환하는 커서(cursor) 객체를 반환합니다. 커서를 반복하여 도큐먼트들을 하나씩 접근할 수 있습니다.

In [25]:
# 여기에 코드를 입력하세요.
result = list(movies.find({'year':1920}))
display(pd.DataFrame(result))

,_id,plot,genres,runtime,rated,cast,num_mflix_comments,poster,title,fullplot,...,released,directors,writers,awards,lastupdated,year,imdb,countries,type,tomatoes
0,573a1391f29313caabcd6d40,A tipsy doctor encounters his patient sleepwal...,"[Comedy, Short]",26,PASSED,"[Harold Lloyd, Roy Brooks, Mildred Davis, Wall...",3.0,https://m.media-amazon.com/images/M/MV5BODliMj...,High and Dizzy,"After a long wait, a young doctor finally has ...",...,1920-07-11,[Hal Roach],"[Frank Terry (story), H.M. Walker (titles)]","{'wins': 0, 'nominations': 1, 'text': '1 nomin...",2015-08-11 00:35:33.717000000,1920,"{'rating': 7, 'votes': 646, 'id': 11293}",[USA],movie,"{'viewer': {'rating': 3.4, 'numReviews': 30, '..."
1,573a1391f29313caabcd6d90,As Alice and Cora Munro attempt to find their ...,"[Adventure, Drama]",73,NOT RATED,"[Wallace Beery, Barbara Bedford, Alan Roscoe, ...",NaN,https://m.media-amazon.com/images/M/MV5BZmJkZD...,The Last of the Mohicans,As Alice and Cora Munro attempt to find their ...,...,1920-11-21,"[Clarence Brown, Maurice Tourneur]","[James Fenimore Cooper (novel), Robert Dillon ...","{'wins': 1, 'nominations': 0, 'text': '1 win.'}",2015-07-19 00:12:27.010000000,1920,"{'rating': 6.9, 'votes': 773, 'id': 11387}",[USA],movie,"{'viewer': {'rating': 3.5, 'numReviews': 306, ..."
2,573a1391f29313caabcd6e2a,A newly wedded couple attempt to build a house...,"[Short, Comedy]",25,TV-G,"[Buster Keaton, Sybil Seely]",1.0,NaN,One Week,Buster and Sybil exit a chapel as newlyweds. A...,...,1920-09-01,"[Edward F. Cline, Buster Keaton]",NaN,"{'wins': 1, 'nominations': 0, 'text': '1 win.'}",2015-05-07 01:07:01.633000000,1920,"{'rating': 8.3, 'votes': 3942, 'id': 11541}",[USA],movie,"{'viewer': {'rating': 4.3, 'numReviews': 752, ..."
3,573a1391f29313caabcd6ea2,The simple-minded son of a rich financier must...,[Comedy],77,NaN,"[Edward Jobson, Beulah Booker, Edward Connelly...",2.0,https://m.media-amazon.com/images/M/MV5BZDNiOD...,The Saphead,Nick Van Alstyne owns the Henrietta silver min...,...,1920-09-01,"[Herbert Blachè, Winchell Smith]","[Bronson Howard (original play ""The Henrietta""...","{'wins': 0, 'nominations': 1, 'text': '1 nomin...",2015-06-20 00:38:08.303000000,1920,"{'rating': 6.2, 'votes': 1020, 'id': 11652}",[USA],movie,"{'viewer': {'rating': 3.3, 'numReviews': 435, ..."
4,573a1391f29313caabcd6f8f,"Abandoned by her fiancè, an educated negro wom...","[Drama, Romance]",79,NaN,"[Evelyn Preer, Flo Clements, James D. Ruffin, ...",NaN,NaN,Within Our Gates,Southern negro Sylvia Landry visits her cousin...,...,1920-01-12,[Oscar Micheaux],[Oscar Micheaux],"{'wins': 1, 'nominations': 0, 'text': '1 win.'}",2015-08-21 00:15:06.017000000,1920,"{'rating': 6.3, 'votes': 1138, 'id': 11870}",[USA],movie,"{'viewer': {'rating': 3.4, 'numReviews': 391, ..."
5,573a139ff29313caabd01709,NaN,[Western],80,NaN,"[Em-koy-e-tie, Hunting Horse, Esther LeBarre, ...",NaN,NaN,The Daughter of Dawn,NaN,...,1920-10-01,[Norbert A. Myles],"[Richard Banks (story), Norbert A. Myles]","{'wins': 1, 'nominations': 0, 'text': '1 win.'}",2015-09-03 00:33:47.333000000,1920,"{'rating': 6.4, 'votes': 30, 'id': 191066}",[USA],movie,"{'viewer': {'rating': 0, 'numReviews': 2}, 'la..."


## Projection (프로젝션)

프로젝션은 `find()` 또는 `find_one()` 메서드 사용 시 결과 도큐먼트에서 원하는 필드만 선택하여 가져오는 기능입니다.

두 번째 인자로 딕셔너리를 전달하여 필드를 포함하거나 제외할 수 있습니다.
`_id` 필드를 제외하려면 `_id: 0`으로 설정하면 됩니다.

In [29]:
# 여기에 코드를 입력하세요.
result = movies.find({'year':1920},{'runtime':1,'rated':1,'title':1,'_id':0})
for doc in result:
    display(pd.DataFrame([doc]))

,runtime,rated,title
0,26,PASSED,High and Dizzy


,runtime,rated,title
0,73,NOT RATED,The Last of the Mohicans


,runtime,title,rated
0,25,One Week,TV-G


,runtime,title
0,77,The Saphead


,runtime,title
0,79,Within Our Gates


,runtime,title
0,80,The Daughter of Dawn


## `sort()` & `limit()`

### `sort()`: 결과 정렬

`sort()` 메서드는 조회 결과를 특정 필드를 기준으로 정렬.
`1`: 오름차순 / `-1`: 내림차순

### `limit()`: 결과 개수 제한

`limit()` 메서드는 조회 결과의 최대 개수를 제한함.

In [32]:
# 여기에 코드를 입력하세요.
result = movies.find({},{"title":1,"_id":0,"year":1}).sort("year",-1).limit(10)
display(pd.DataFrame(result))

,title,year
0,The Saboteurs,2015è
1,Halo: Nightfall,2014è
2,The Roosevelts: An Intimate History,2014è
3,Hit & Miss,2012è
4,Penance,2012è
5,The Weight of the Nation,2012è
6,Vietnam in HD,2011è
7,Falling Skies,2011è
8,Stephen Hawking's Universe,2010è
9,Third Reich: The Rise & Fall,2010è


## 도큐먼트 업데이트

### `update_one()`: 단일 도큐먼트 업데이트

`update_one()` 메서드는 주어진 조건과 일치하는 첫 번째 도큐먼트를 업데이트합니다.

`$set` 연산자를 사용하여 필드를 설정할 수 있습니다.

In [39]:
# 여기에 코드를 입력하세요.
pre = movies.find_one({"title":"The Ace of Hearts"},{"hah":1,"year":1,"title":1,"_id":0})
print(pre)
print("수정전")
result = movies.update_one(
    {"title":"The Ace of Hearts"},
    {
        "$set":{"hah":"skrRR"}
    }
)
print("2021 -> 2020 수정완료")
print(f"매치된 도큐먼트 수: {result.matched_count}")
print(f"수정된 도큐먼트 수: {result.modified_count}")
print("수정후")
post = movies.find_one({"title":"The Ace of Hearts"},{"hah":1,"year":1,"title":1,"_id":0})
display(post)

{'title': 'The Ace of Hearts', 'year': 2020}
수정전
2021 -> 2020 수정완료
매치된 도큐먼트 수: 1
수정된 도큐먼트 수: 1
수정후


{'title': 'The Ace of Hearts', 'year': 2020, 'hah': 'skrRR'}

### `update_many()`: 여러 도큐먼트 업데이트

`update_many()` 메서드는 주어진 조건과 일치하는 모든 도큐먼트를 업데이트합니다.

In [ ]:
# 여기에 코드를 입력하세요.
result = movies.update_many(
    {"title":"The Ace of Hearts"},
    {
    "$set":{"hah":"skrRR"}
    }
)

## 도큐먼트 삭제

### `delete_one()`: 단일 도큐먼트 삭제

`delete_one()` 메서드는 주어진 조건과 일치하는 첫 번째 도큐먼트를 삭제합니다.

In [40]:
# 여기에 코드를 입력하세요.
result = movies.delete_one({"title":"The Ace of Hearts"})
print(f"삭제된 doc수{result.deleted_count}")



삭제된 doc수1


0

## 컬렉션 삭제

### `drop()`: 컬렉션 전체 삭제

`drop()` 메서드는 컬렉션 자체를 완전히 삭제합니다. 이 작업은 되돌릴 수 없으므로 주의해야 합니다.

In [41]:
temp_collection.drop()
print('temp_movies'in db.list_collection_names())

False


## Aggregation

`Aggregation Pipeline`

여러 단계(stage)를 순차적으로 거치면서 데이터를 변환하고 분석하는 MongoDB의 강력한 기능입니다.
SQL의 GROUP BY, HAVING, ORDER BY, 데이터 전처리 기능을 합친 것과 비슷합니다.

### `$match`
`$match` 단계는 조건에 맞는 도큐먼트만 다음 단계로 전달합니다.
SQL의 WHERE 절과 비슷하며, 필터링 시 가장 먼저 사용하는 단계입니다.

$match는 MongoDB의 다양한 비교 연산자를 함께 사용할 수 있습니다.

- `$eq`  : 같다 (=)

- `$gte`: 크거나 같다 (>=)

- `$lte`: 작거나 같다 (<=)

- `$gt` : 크다 (>)

- `$lt` : 작다 (<)

- `$ne` : 같지 않다 (!=)

- `$in` : 여러 값 중 일치하는 것 조회

- `$nin` : 배열 안에 없는 경우

In [47]:
# 여기에 코드를 입력하세요.
res_1 = movies.aggregate([
    {
        "$match":{
            "year":{"$gte":2001},
            "imdb.rating":{"$gt":7}
        }
    },
    {
        "$limit":5
    }
])
display(list(res_1))


[{'_id': ObjectId('573a139af29313caabcf0ea6'),
  'fullplot': '"Frida" chronicles the life Frida Kahlo shared unflinchingly and openly with Diego Rivera, as the young couple took the art world by storm. From her complex and enduring relationship with her mentor and husband to her illicit and controversial affair with Leon Trotsky, to her provocative and romantic entanglements with women, Frida Kahlo lived a bold and uncompromising life as a political, artistic, and sexual revolutionary.',
  'imdb': {'rating': 7.4, 'votes': 57256, 'id': 120679},
  'year': 2002,
  'plot': 'A biography of artist Frida Kahlo, who channeled the pain of a crippling injury and her tempestuous marriage into her work.',
  'genres': ['Biography', 'Drama', 'Romance'],
  'rated': 'R',
  'metacritic': 61,
  'title': 'Frida',
  'lastupdated': '2015-09-12 00:33:36.510000000',
  'languages': ['English', 'French', 'Russian'],
  'writers': ['Hayden Herrera (book)',
   'Clancy Sigal (screenplay)',
   'Diane Lake (screenpl

In [49]:
res_2 = movies.aggregate([
    {
        "$match":{
            "year":{"$gte":2000,"$lte":2009},
            "imdb.rating":{"$gt":7},
            "genres":{"$in":["Mystery"]}
        }
    },
    {"$limit":5}
])
display(list(res_2))

[{'_id': ObjectId('573a139ef29313caabcfbc36'),
  'plot': 'After a car wreck on the winding Mulholland Drive renders a woman amnesiac, she and a perky Hollywood-hopeful search for clues and answers across Los Angeles in a twisting venture beyond dreams and reality.',
  'genres': ['Drama', 'Mystery', 'Thriller'],
  'runtime': 147,
  'metacritic': 81,
  'rated': 'R',
  'cast': ['Naomi Watts', 'Jeanne Bates', 'Dan Birnbaum', 'Laura Harring'],
  'poster': 'https://m.media-amazon.com/images/M/MV5BNWM2MDZmMDgtYjViOS00YzBmLWE4YzctMDMyYTQ2YTc4MmVkXkEyXkFqcGdeQXVyNDk3NzU2MTQ@._V1_SY1000_SX677_AL_.jpg',
  'title': 'Mulholland Drive',
  'fullplot': 'A bright-eyed young actress travels to Hollywood, only to be ensnared in a dark conspiracy involving a woman who was nearly murdered, and now has amnesia because of a car crash. Eventually, both women are pulled into a psychotic illusion involving a dangerous blue box, a director named Adam Kesher, and the mysterious night club Silencio.',
  'languages

In [50]:
res_2 = movies.aggregate([
    {
        "$match":{
            "$and":[
                {"year":{"$gte":2000,"$lte":2009}},
                {"imdb.rating":{"$gt":7}},
                {"genres":{"$in":["Mystery"]}}
            ]

        }
    },
    {"$limit":5}
])
display(list(res_2))

[{'_id': ObjectId('573a139ef29313caabcfbc36'),
  'plot': 'After a car wreck on the winding Mulholland Drive renders a woman amnesiac, she and a perky Hollywood-hopeful search for clues and answers across Los Angeles in a twisting venture beyond dreams and reality.',
  'genres': ['Drama', 'Mystery', 'Thriller'],
  'runtime': 147,
  'metacritic': 81,
  'rated': 'R',
  'cast': ['Naomi Watts', 'Jeanne Bates', 'Dan Birnbaum', 'Laura Harring'],
  'poster': 'https://m.media-amazon.com/images/M/MV5BNWM2MDZmMDgtYjViOS00YzBmLWE4YzctMDMyYTQ2YTc4MmVkXkEyXkFqcGdeQXVyNDk3NzU2MTQ@._V1_SY1000_SX677_AL_.jpg',
  'title': 'Mulholland Drive',
  'fullplot': 'A bright-eyed young actress travels to Hollywood, only to be ensnared in a dark conspiracy involving a woman who was nearly murdered, and now has amnesia because of a car crash. Eventually, both women are pulled into a psychotic illusion involving a dangerous blue box, a director named Adam Kesher, and the mysterious night club Silencio.',
  'languages

#### 논리 연산자

`$and`, `$or`, `$not`

`$nor`: 들어 있는 조건을 모두 만족하지 않을 때

In [ ]:
# 여기에 코드를 입력하세요.

### `$group`

$group은 SQL의 GROUP BY처럼 데이터를 특정 기준으로 묶어 집계(Aggregation)를 수행합니다.
MongoDB에서는 그룹 기준이 `_id` 필드이며, 계산된 필드를 새로운 key로 추가할 수 있습니다.

자주 사용하는 집계 연산자:
  - $sum : 합계 (count 용도로도 사용)

  - $avg : 평균

  - $min : 최소값

  - $max : 최대값

주의사항:
  - _id 값을 null로 설정하면 전체 집계가 가능함 (ex: 전체 평균)
  - $group 내부에서는 원본 문서의 필드를 직접 사용하려면 `$필드명` 으로 참조해야 함

In [54]:
# 여기에 코드를 입력하세요.
res_1=movies.aggregate([
    {
        "$group":{
            "_id":"$year",
            "movie_cnt":{"$sum":1}
        }
    }
])

res_2 = movies.aggregate([
    {
        "$group":{
            "_id":"$year",
            "rating_avg":{"$avg":"$imdb.rating"}
        }
    },
    {"$sort":{"rating_avg":-1}}
])
display(list(res_2))

[{'_id': '2006è2007', 'rating_avg': 9.0},
 {'_id': '1995è', 'rating_avg': 9.0},
 {'_id': '1999è', 'rating_avg': 8.9},
 {'_id': '1987è', 'rating_avg': 8.9},
 {'_id': '2007è', 'rating_avg': 8.533333333333333},
 {'_id': '2006è2012', 'rating_avg': 8.5},
 {'_id': '1996è', 'rating_avg': 8.4},
 {'_id': '2003è', 'rating_avg': 8.4},
 {'_id': '2005è', 'rating_avg': 8.4},
 {'_id': '2015è', 'rating_avg': 8.4},
 {'_id': '2009è', 'rating_avg': 8.3},
 {'_id': '1994è1998', 'rating_avg': 8.2},
 {'_id': '2010è', 'rating_avg': 8.2},
 {'_id': '1981è', 'rating_avg': 8.0},
 {'_id': '1988è', 'rating_avg': 7.85},
 {'_id': '2011è', 'rating_avg': 7.75},
 {'_id': '1986è', 'rating_avg': 7.7},
 {'_id': 1925, 'rating_avg': 7.684615384615385},
 {'_id': 1928, 'rating_avg': 7.684210526315789},
 {'_id': 1921, 'rating_avg': 7.68},
 {'_id': 1922, 'rating_avg': 7.6000000000000005},
 {'_id': 1924, 'rating_avg': 7.588888888888889},
 {'_id': 1939, 'rating_avg': 7.586206896551724},
 {'_id': 1927, 'rating_avg': 7.5454545454545

### `$project`

`$project`는 SQL의 SELECT처럼 원하는 필드만 반환하거나 새 필드를 계산하는 단계입니다. 4일차에 진행했던 `projection`과 동일한 개념입니다.

기능:
  - 필드 포함/제외 (1: 포함, 0: 제외)

  - 필드 이름 변경

  - 새로운 계산 필드 생성(산술 연산 가능)

  - 중첩된 필드 접근 (ex. imdb.rating)

주의:
  - `_id`는 기본적으로 항상 포함되므로 제외하려면 `_id: 0` 해야 함


In [57]:
# 여기에 코드를 입력하세요.
result = movies.aggregate([
    {
        "$project":{
            "title":1,
            "score":{
                "$multiply":["$imdb.rating",10]
            }
        }
    },{"$limit":10}
])
display(list(result))

[{'_id': ObjectId('573a1390f29313caabcd4135'),
  'title': 'Blacksmith Scene',
  'score': 62.0},
 {'_id': ObjectId('573a1390f29313caabcd42e8'),
  'title': 'The Great Train Robbery',
  'score': 74.0},
 {'_id': ObjectId('573a1390f29313caabcd4323'),
  'title': 'The Land Beyond the Sunset',
  'score': 71.0},
 {'_id': ObjectId('573a1390f29313caabcd446f'),
  'title': 'A Corner in Wheat',
  'score': 66.0},
 {'_id': ObjectId('573a1390f29313caabcd4803'),
  'title': 'Winsor McCay, the Famous Cartoonist of the N.Y. Herald and His Moving Comics',
  'score': 73.0},
 {'_id': ObjectId('573a1390f29313caabcd4eaf'),
  'title': 'Traffic in Souls',
  'score': 60},
 {'_id': ObjectId('573a1390f29313caabcd50e5'),
  'title': 'Gertie the Dinosaur',
  'score': 73.0},
 {'_id': ObjectId('573a1390f29313caabcd516c'),
  'title': 'In the Land of the Head Hunters',
  'score': 58.0},
 {'_id': ObjectId('573a1390f29313caabcd5293'),
  'title': 'The Perils of Pauline',
  'score': 76.0},
 {'_id': ObjectId('573a1390f29313caab